# V84 - Cross-encoder finetune screen

Pre-registered plan: zero-shot cross-attention is dead, but no version ever adapted the cross-encoder to graded relevance. Finetune MiniLM (33M params, fits local CUDA) with a regression head on user-syllabus pairs, then a fold-1 screen against Ridge on the same fold. Full OOF only starts if the screen is positive. MSE on graded scores, one epoch first for a cheap signal check, up to three if the loss moves sensibly.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sentence_transformers import CrossEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.model_selection import StratifiedGroupKFold
import re

SEED = 3333
GATE_FLOOR = 0.002
MODEL = "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1"
rng = np.random.RandomState(SEED)

CANDIDATES = [Path("/kaggle/input/intelligo-data/data"), Path("data")]
DATA = next((p for p in CANDIDATES if (p / "train_relevance.csv").exists()), None)
assert DATA is not None, "train_relevance.csv not found"
OUT = Path("/kaggle/working" if Path("/kaggle/working").exists() else "results/v84")
OUT.mkdir(parents=True, exist_ok=True)
assert torch.cuda.is_available(), "v84 needs CUDA"
print("DATA:", DATA.resolve())

DATA: /home/athilla/Documents/IF_ITB/Lomba/IT-TODAY/intelligo-course-recommender/data


## Finetune on fold-1 train pairs, screen on fold-1 validation

Pairs are user chat plus syllabus with graded targets. Ridge on word-only is the same-fold reference.

In [2]:
MODS = [f"M_{i:03d}" for i in range(1, 18)]
train = pd.read_csv(DATA / "train_relevance.csv")
chats = pd.read_csv(DATA / "chat_history.csv")
catalog = pd.read_csv(DATA / "modules_catalog.csv")
Y = train[MODS].to_numpy(float)
n_train = len(train)
by_user = chats.groupby("user_id")["user_chat_text"].apply(list)
docs = [" ".join(str(t) for t in by_user.get(u, []))[:1000] for u in list(train["user_id"])]
syll = [str(catalog.set_index("module_id").loc[m, "description_and_syllabus"])[:1000] if m in set(catalog["module_id"]) else "" for m in MODS]

def template_group(texts):
    if not texts:
        return "NO_CHAT"
    return re.sub(r"\d+", "#", " ".join(str(texts[0]).lower().split()[:6]))
fold_groups = np.array([template_group(by_user.get(u, [])) for u in train["user_id"]])
strata = Y.argmax(1)

def ndcg5(yt, ys):
    order = np.argsort(-ys, axis=1)
    rel = np.take_along_axis(yt, order, 1)[:, :5]
    disc = 1.0 / np.log2(np.arange(2, 7))
    dcg = (rel * disc).sum(1)
    ideal = np.sort(yt, axis=1)[:, ::-1][:, :5]
    idcg = (ideal * disc).sum(1)
    return np.where(idcg > 0, dcg / np.maximum(idcg, 1e-12), 0.0)

W = TfidfVectorizer(max_features=5000, ngram_range=(1, 1), min_df=3).fit_transform(docs).toarray()
skf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
folds = list(skf.split(W, strata, fold_groups))
tr1 = np.concatenate([folds[k][0] for k in range(1, 5)])
va1 = folds[0][1]
oof_w = np.zeros((len(va1), len(MODS)))
for j in range(len(MODS)):
    m = Ridge(alpha=1.0)
    m.fit(W[tr1], Y[tr1, j])
    oof_w[:, j] = m.predict(W[va1])
ref = ndcg5(Y[va1], oof_w).mean()
print(f"fold-1 ridge ref: {ref:.5f}")

from torch.utils.data import DataLoader
from sentence_transformers import InputExample
tr1u = np.unique(np.concatenate([folds[k][0] for k in range(1, 5)]))
examples = [InputExample(texts=[docs[u], syll[j]], label=float(Y[u, j])) for u in tr1u for j in range(len(MODS))]
print(f"train pairs: {len(examples)}")
ce = CrossEncoder(MODEL, num_labels=1, max_length=128, device='cuda')
ce.fit(train_dataloader=DataLoader(examples, shuffle=True, batch_size=32), epochs=2, warmup_steps=500, use_amp=True, show_progress_bar=True)
va_pairs = [(docs[u], syll[j]) for u in va1 for j in range(len(MODS))]
pred = np.array(ce.predict(va_pairs, batch_size=64, show_progress_bar=True), dtype=np.float32).reshape(len(va1), len(MODS))
ce.save(str(OUT / 'ce_ft'))
got = ndcg5(Y[va1], pred).mean()
print(f"fold-1 ce-finetune: {got:.5f} delta={got - ref:+.5f}")
print('SCREEN PASS: run full OOF next' if got - ref > 0 else 'SCREEN FAILED: finetuned cross-attention adds nothing')


fold-1 ridge ref: 0.68268
train pairs: 68000


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Step,Training Loss
500,0.579487
1000,0.468536
1500,0.449758
2000,0.449666
2500,0.440912
3000,0.433537
3500,0.430445
4000,0.429455


Batches:   0%|          | 0/288 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

fold-1 ce-finetune: 0.63822 delta=-0.04447
SCREEN FAILED: finetuned cross-attention adds nothing
